##Feature Engineering und zeitbasierte Kreuzvalidierung

Dieses Notebook führt das Feature Engineering für die anschließende Machine-Learning-Modellierung zur Prognose bzw. Klassifikation von Rezessionsphasen durch. Ausgangspunkt ist der in Notebook 3 vollständig vorverarbeitete Datensatz.

Zunächst werden für die makroökonomischen Prädiktorvariablen zeitverzögerte Merkmale (Lag Features) von 1 bis 12 Monaten erzeugt, um vergangene Informationen der Indikatoren für die Modellierung verfügbar zu machen. Anschließend werden die erzeugten Features hinsichtlich Vollständigkeit, zeitlicher Ordnung und Konsistenz der Zielvariable Rezession überprüft.

Darauf aufbauend werden die finale Feature-Matrix X_final und die Zielvariable y_final vorbereitet. Für eine zeitgerechte Modellevaluation werden außerdem die historischen Rezessionsperioden identifiziert und eine benutzerdefinierte zeitbasierte Kreuzvalidierung mit expandierendem Trainingsfenster (Expanding Window) konstruiert. Dabei wird sichergestellt, dass die Trainingsdaten zeitlich vor den jeweiligen Testperioden liegen und somit die chronologische Struktur der Zeitreihe berücksichtigt wird.

**Import und Laden des vorverarbeiteten Datensatzes**

Zu Beginn wird der in Notebook 3 vollständig vorverarbeitete Datensatz Final_Dataset_ML_1991_2026_preprocessed.xlsx eingelesen. Die Datumsspalte wird in ein einheitliches Datumsformat konvertiert und der Datensatz anschließend chronologisch sortiert. Dies ist insbesondere für das nachfolgende Lag Feature Engineering erforderlich, da die Erstellung zeitverzögerter Variablen eine korrekte zeitliche Reihenfolge der Beobachtungen voraussetzt. Abschließend werden die Dimension, der Untersuchungszeitraum sowie die ersten Beobachtungen des Datensatzes kontrolliert.

In [30]:
import pandas as pd

# --- Datensatz laden ---

file_name = "Final_Dataset_ML_1991_2026_preprocessed.xlsx"

try:
    df = pd.read_excel(file_name)

except FileNotFoundError:
    print(f"Fehler: '{file_name}' wurde nicht gefunden.")
    print("Bitte die Excel-Datei zuerst manuell in Colab hochladen.")
    df = None

# --- Grundkontrolle ---

if df is not None:
    df["Datum"] = pd.to_datetime(df["Datum"])
    df = df.sort_values("Datum").reset_index(drop=True)

    print("--- Datensatz erfolgreich geladen ---")
    print("Dimension:", df.shape)
    print("Zeitraum:", df["Datum"].min(), "bis", df["Datum"].max())

    display(df.head())

--- Datensatz erfolgreich geladen ---
Dimension: (413, 30)
Zeitraum: 1992-01-31 00:00:00 bis 2026-05-31 00:00:00


,Datum,Industrial_Production,Manufacturing_Output,Capital_Goods_Output,Intermediate_Goods_Output,Consumer_Goods_Output,Orders_Abroad_Intermediate_Capital,Construction_Orders,Unemployment_Rate,Employment,...,Domestic_Intermediate_Goods_Orders,Euribor_3M,Brent_Oil,Production_Expectations_Manufacturing,Business_Situation_Services,Business_Situation_Retail,Term_Spread_Germany,Exports,Imports,Rezession
0,1992-01-31,0.016159,0.016487,0.019311,0.006873,0.018915,-0.081346,0.054115,0.3,-0.000621,...,0.008299,-0.05,-0.013673,2.1,-0.938580,0.0,-0.2150,-0.028073,0.047531,0
1,1992-02-29,0.009816,0.012500,0.016261,0.010899,0.001970,0.008439,0.074848,-0.2,-0.000777,...,-0.021722,0.08,-0.006076,0.4,6.598765,-6.2,-0.1075,0.008749,-0.010206,0
2,1992-03-31,-0.023472,-0.028988,-0.036965,-0.020535,-0.023906,0.022162,-0.031749,0.0,-0.001193,...,0.018411,0.09,-0.023544,-0.6,-7.300699,-11.3,0.0175,0.015558,-0.012720,1
3,1992-04-30,-0.001251,-0.001280,0.004175,-0.006940,0.010030,-0.050573,-0.020051,0.1,-0.000805,...,-0.038889,0.05,0.070618,-1.7,-4.267983,0.0,-0.0425,0.021046,0.042676,1
4,1992-05-31,-0.011328,-0.009003,-0.008368,-0.001394,-0.017112,-0.020379,-0.020461,0.1,-0.000728,...,-0.001726,0.04,0.049998,-1.7,-2.872531,-3.7,0.0000,-0.094706,-0.061970,1


**Finale Ausgangskontrolle**

Vor der Erstellung neuer Features wird die Qualität des importierten Datensatzes erneut kontrolliert. Dabei werden fehlende Werte, Duplikate, die chronologische Reihenfolge sowie die Vollständigkeit und Klassenverteilung der Zielvariable Rezession überprüft. Zusätzlich werden Dimension und Untersuchungszeitraum dokumentiert.

In [31]:
print("--- Ausgangskontrolle vor dem Feature Engineering ---")

# Fehlende Werte
print("\nGesamtzahl fehlender Werte:")
print(df.isna().sum().sum())

# Duplikate
print("\nVollständig doppelte Zeilen:")
print(df.duplicated().sum())

print("\nDoppelte Datumswerte:")
print(df["Datum"].duplicated().sum())

# Chronologische Reihenfolge
print("\nChronologisch sortiert:")
print(df["Datum"].is_monotonic_increasing)

# Zielvariable
print("\nKlassenverteilung der Zielvariable Rezession:")
print(df["Rezession"].value_counts().sort_index())

print("\nFehlende Werte in Rezession:")
print(df["Rezession"].isna().sum())

# Dimension und Zeitraum
print("\nDimension:", df.shape)
print("Zeitraum:", df["Datum"].min(), "bis", df["Datum"].max())

--- Ausgangskontrolle vor dem Feature Engineering ---

Gesamtzahl fehlender Werte:
0

Vollständig doppelte Zeilen:
0

Doppelte Datumswerte:
0

Chronologisch sortiert:
True

Klassenverteilung der Zielvariable Rezession:
Rezession
0    325
1     88
Name: count, dtype: int64

Fehlende Werte in Rezession:
0

Dimension: (413, 30)
Zeitraum: 1992-01-31 00:00:00 bis 2026-05-31 00:00:00


**Lag Feature Engineering**

Zur Berücksichtigung der zeitlichen Pfadabhängigkeit makroökonomischer Indikatoren und zur Abbildung potenzieller Vorlaufbeziehungen wird der Merkmalsraum durch die Konstruktion zeitverzögerter Prädiktoren (Lag Features) erweitert. Hierfür werden für alle Prädiktorvariablen Verzögerungen von t−1 bis t−12 Monaten generiert. Dadurch stehen den Machine-Learning-Modellen Informationen über die historische Entwicklung der makroökonomischen Indikatoren zur Verfügung, die zur Klassifikation der aktuellen Konjunkturphase genutzt werden können. Die Zielvariable Rezession sowie die Zeitvariable Datum werden von der Lag-Transformation ausgeschlossen.

Durch die zeitliche Verschiebung entstehen für die ersten zwölf Monate technisch bedingt fehlende Werte, da für diese Beobachtungen keine vollständige zwölfmonatige Historie verfügbar ist. Diese Zeilen werden anschließend entfernt und der Index des Datensatzes neu gesetzt. Abschließend werden die Dimensionen, die ersten und letzten Beobachtungen sowie die Datentypen des erzeugten Lag-Datensatzes kontrolliert.

In [32]:
# Combine X and y back with date for easier lagging operations
# We'll re-separate them later if needed
df_for_lagging = pd.concat([date_column, X, y], axis=1)

# Initialize a list to hold all the dataframes with lagged features
lagged_features_dfs = [df_for_lagging[['Datum', target_column_name]]]

# Identify predictor columns for lagging (all columns in X)
predictor_columns = X.columns.tolist()

# Create lagged features for each predictor variable from Lag 1 to Lag 12
for lag in range(1, 13):  # Lag 1 to Lag 12
    lagged_df_temp = df_for_lagging[predictor_columns].shift(lag)
    lagged_df_temp.columns = [f'{col}_lag_{lag}' for col in predictor_columns]
    lagged_features_dfs.append(lagged_df_temp)

# Concatenate all lagged features with the original 'Datum' and 'Rezession'
# The original df_for_lagging will be used as the base, and lagged features will be added
final_df_lagged = pd.concat(lagged_features_dfs, axis=1)

print(f"DataFrame shape after creating lagged features (before dropping NaNs): {final_df_lagged.shape}")

# Remove rows with missing values (NaNs) introduced by lagging
# This effectively removes the first 12 rows which would have NaNs due to shifting
initial_rows_before_nan_drop = final_df_lagged.shape[0]
final_df_lagged.dropna(inplace=True)
final_df_lagged.reset_index(drop=True, inplace=True)

num_rows_dropped = initial_rows_before_nan_drop - final_df_lagged.shape[0]
print(f"Number of rows dropped due to NaN values (from lagging): {num_rows_dropped}")

# Verify the final dataset
print(f"\nFinal DataFrame shape after lagging and dropping NaNs: {final_df_lagged.shape}")
print("\nFirst 5 rows of the final lagged DataFrame:")
display(final_df_lagged.head())
print("\nLast 5 rows of the final lagged DataFrame:")
display(final_df_lagged.tail())

print("\nData types of the final lagged DataFrame:")
final_df_lagged.info()


DataFrame shape after creating lagged features (before dropping NaNs): (413, 338)
Number of rows dropped due to NaN values (from lagging): 12

Final DataFrame shape after lagging and dropping NaNs: (401, 338)

First 5 rows of the final lagged DataFrame:


,Datum,Rezession,Industrial_Production_lag_1,Manufacturing_Output_lag_1,Capital_Goods_Output_lag_1,Intermediate_Goods_Output_lag_1,Consumer_Goods_Output_lag_1,Orders_Abroad_Intermediate_Capital_lag_1,Construction_Orders_lag_1,Unemployment_Rate_lag_1,...,Domestic_Capital_Goods_Orders_lag_12,Domestic_Intermediate_Goods_Orders_lag_12,Euribor_3M_lag_12,Brent_Oil_lag_12,Production_Expectations_Manufacturing_lag_12,Business_Situation_Services_lag_12,Business_Situation_Retail_lag_12,Term_Spread_Germany_lag_12,Exports_lag_12,Imports_lag_12
0,1993-01-31,1,-0.018792,-0.017894,-0.035307,-0.010394,0.005206,-0.015456,0.101063,0.1,...,-0.030191,0.008299,-0.05,-0.013673,2.1,-0.938580,0.0,-0.2150,-0.028073,0.047531
1,1993-02-28,1,-0.001356,-0.006969,-0.017337,-0.012012,-0.011488,-0.015699,-0.062370,0.1,...,0.022728,-0.021722,0.08,-0.006076,0.4,6.598765,-6.2,-0.1075,0.008749,-0.010206
2,1993-03-31,1,-0.016416,-0.018349,-0.027399,-0.012158,-0.012685,0.006309,-0.024218,0.1,...,-0.024646,0.018411,0.09,-0.023544,-0.6,-7.300699,-11.3,0.0175,0.015558,-0.012720
3,1993-04-30,1,-0.001380,0.001423,0.009756,-0.003063,0.005305,-0.015848,0.043827,0.1,...,-0.029213,-0.038889,0.05,0.070618,-1.7,-4.267983,0.0,-0.0425,0.021046,0.042676
4,1993-05-31,1,-0.008322,-0.007138,-0.011391,-0.001535,-0.010638,0.012699,-0.019608,0.1,...,-0.040328,-0.001726,0.04,0.049998,-1.7,-2.872531,-3.7,0.0000,-0.094706,-0.061970



Last 5 rows of the final lagged DataFrame:


,Datum,Rezession,Industrial_Production_lag_1,Manufacturing_Output_lag_1,Capital_Goods_Output_lag_1,Intermediate_Goods_Output_lag_1,Consumer_Goods_Output_lag_1,Orders_Abroad_Intermediate_Capital_lag_1,Construction_Orders_lag_1,Unemployment_Rate_lag_1,...,Domestic_Capital_Goods_Orders_lag_12,Domestic_Intermediate_Goods_Orders_lag_12,Euribor_3M_lag_12,Brent_Oil_lag_12,Production_Expectations_Manufacturing_lag_12,Business_Situation_Services_lag_12,Business_Situation_Retail_lag_12,Term_Spread_Germany_lag_12,Exports_lag_12,Imports_lag_12
396,2026-01-31,0,-0.009767,-0.011809,-0.021783,-0.007160,0.007555,0.029050,-0.034906,0.0,...,-0.054899,0.022114,-0.121,0.070688,-10.9,0.3,-4.3,-0.031115,0.008335,0.033959
397,2026-02-28,0,-0.001091,-0.006501,-0.011072,-0.004802,-0.004310,-0.085181,-0.061028,0.0,...,-0.006795,-0.013456,-0.179,-0.049522,-6.9,2.1,1.4,0.132600,0.009408,-0.000967
398,2026-03-31,0,0.000000,0.000000,-0.001013,0.005999,-0.010858,0.057223,0.058873,0.0,...,0.014665,0.014670,-0.083,-0.036584,-5.2,-5.5,-2.5,-0.082732,0.013986,-0.003524
399,2026-04-30,0,-0.007671,-0.005450,-0.011207,0.004773,-0.013187,0.048946,-0.002160,0.0,...,0.036288,-0.033316,-0.193,-0.065336,-6.4,1.3,0.2,0.190552,-0.010187,0.014806
400,2026-05-31,0,0.000000,0.000000,-0.015488,0.014185,0.018631,-0.035945,0.000000,0.1,...,-0.022939,-0.031870,-0.162,-0.055528,-1.7,-1.3,4.6,0.074259,-0.014207,-0.033695



Data types of the final lagged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Columns: 338 entries, Datum to Imports_lag_12
dtypes: datetime64[ns](1), float64(336), int64(1)
memory usage: 1.0 MB


**Qualitätskontrolle der erzeugten Lag Features**

Nach der Erstellung der zeitverzögerten Prädiktoren wird eine umfassende Qualitätskontrolle des Lag-Datensatzes durchgeführt. Zunächst wird überprüft, ob nach der Entfernung der durch das Lagging entstandenen Anfangsbeobachtungen noch fehlende Werte vorhanden sind. Anschließend wird kontrolliert, ob die tatsächliche Anzahl der erzeugten Spalten der erwarteten Anzahl von zwölf Lag-Features je ursprünglicher Prädiktorvariable zuzüglich der Variablen Datum und Rezession entspricht.

Darüber hinaus wird die zeitliche Integrität des Datensatzes überprüft, indem die chronologische Sortierung der Datumsspalte sowie der resultierende Untersuchungszeitraum kontrolliert werden. Zusätzlich wird die Zielvariable Rezession hinsichtlich ihrer Klassenverteilung und möglicher fehlender Werte untersucht. Abschließend ermöglichen die ersten und letzten Beobachtungen des Datensatzes eine visuelle Plausibilitätskontrolle der erzeugten Lag-Struktur.

In [33]:
print("--- Qualitätskontrolle der erzeugten Lag Features ---")

# 1. Überprüfung auf fehlende Werte (NaNs) nach dem Droppen
print("\nGesamtzahl fehlender Werte im final_df_lagged:")
missing_values_count = final_df_lagged.isna().sum().sum()
print(missing_values_count)
if missing_values_count == 0:
    print("Keine fehlenden Werte im final_df_lagged vorhanden.")
else:
    print("Achtung: Es sind noch fehlende Werte vorhanden. Bitte überprüfen.")

# 2. Überprüfung der Dimensionen des DataFrames
original_predictor_count = len(X.columns) # X is the original predictor DataFrame
expected_columns = (original_predictor_count * 12) + 2 # 12 lags + Datum + Rezession

print(f"\nDimension des final_df_lagged: {final_df_lagged.shape}")
print(f"Erwartete Anzahl an Spalten: {expected_columns}")
if final_df_lagged.shape[1] == expected_columns:
    print("Anzahl der Spalten stimmt mit den erwarteten Lag-Features überein.")
else:
    print( "Achtung: Anzahl der Spalten weicht von der Erwartung ab.")

# 3. Überprüfung der Datumsintegrität und Reihenfolge
print("\nChronologische Sortierung der 'Datum'-Spalte:")
is_monotonic = final_df_lagged['Datum'].is_monotonic_increasing
print(is_monotonic)
if is_monotonic:
    print("'Datum'-Spalte ist chronologisch sortiert.")
else:
    print("Achtung: 'Datum'-Spalte ist nicht chronologisch sortiert.")

print(f"Zeitraum im final_df_lagged: {final_df_lagged['Datum'].min()} bis {final_df_lagged['Datum'].max()}")

# 4. Überprüfung der Zielvariable 'Rezession' im final_df_lagged
print("\nKlassenverteilung der Zielvariable 'Rezession' im final_df_lagged:")
print(final_df_lagged['Rezession'].value_counts().sort_index())
print(f"Fehlende Werte in 'Rezession' (final_df_lagged): {final_df_lagged['Rezession'].isna().sum()}")
if final_df_lagged['Rezession'].isna().sum() == 0:
    print("Keine fehlenden Werte in der Zielvariable 'Rezession'.")
else:
    print("Achtung: Fehlende Werte in der Zielvariable 'Rezession' vorhanden.")

# 5. Anzeigen der ersten und letzten Zeilen zur visuellen Kontrolle
print("\nErste 5 Zeilen des final_df_lagged:")
display(final_df_lagged.head())
print("\nLetzte 5 Zeilen des final_df_lagged:")
display(final_df_lagged.tail())

print("\n--- Qualitätskontrolle abgeschlossen ---")

--- Qualitätskontrolle der erzeugten Lag Features ---

Gesamtzahl fehlender Werte im final_df_lagged:
0
Keine fehlenden Werte im final_df_lagged vorhanden.

Dimension des final_df_lagged: (401, 338)
Erwartete Anzahl an Spalten: 338
Anzahl der Spalten stimmt mit den erwarteten Lag-Features überein.

Chronologische Sortierung der 'Datum'-Spalte:
True
'Datum'-Spalte ist chronologisch sortiert.
Zeitraum im final_df_lagged: 1993-01-31 00:00:00 bis 2026-05-31 00:00:00

Klassenverteilung der Zielvariable 'Rezession' im final_df_lagged:
Rezession
0    323
1     78
Name: count, dtype: int64
Fehlende Werte in 'Rezession' (final_df_lagged): 0
Keine fehlenden Werte in der Zielvariable 'Rezession'.

Erste 5 Zeilen des final_df_lagged:


,Datum,Rezession,Industrial_Production_lag_1,Manufacturing_Output_lag_1,Capital_Goods_Output_lag_1,Intermediate_Goods_Output_lag_1,Consumer_Goods_Output_lag_1,Orders_Abroad_Intermediate_Capital_lag_1,Construction_Orders_lag_1,Unemployment_Rate_lag_1,...,Domestic_Capital_Goods_Orders_lag_12,Domestic_Intermediate_Goods_Orders_lag_12,Euribor_3M_lag_12,Brent_Oil_lag_12,Production_Expectations_Manufacturing_lag_12,Business_Situation_Services_lag_12,Business_Situation_Retail_lag_12,Term_Spread_Germany_lag_12,Exports_lag_12,Imports_lag_12
0,1993-01-31,1,-0.018792,-0.017894,-0.035307,-0.010394,0.005206,-0.015456,0.101063,0.1,...,-0.030191,0.008299,-0.05,-0.013673,2.1,-0.938580,0.0,-0.2150,-0.028073,0.047531
1,1993-02-28,1,-0.001356,-0.006969,-0.017337,-0.012012,-0.011488,-0.015699,-0.062370,0.1,...,0.022728,-0.021722,0.08,-0.006076,0.4,6.598765,-6.2,-0.1075,0.008749,-0.010206
2,1993-03-31,1,-0.016416,-0.018349,-0.027399,-0.012158,-0.012685,0.006309,-0.024218,0.1,...,-0.024646,0.018411,0.09,-0.023544,-0.6,-7.300699,-11.3,0.0175,0.015558,-0.012720
3,1993-04-30,1,-0.001380,0.001423,0.009756,-0.003063,0.005305,-0.015848,0.043827,0.1,...,-0.029213,-0.038889,0.05,0.070618,-1.7,-4.267983,0.0,-0.0425,0.021046,0.042676
4,1993-05-31,1,-0.008322,-0.007138,-0.011391,-0.001535,-0.010638,0.012699,-0.019608,0.1,...,-0.040328,-0.001726,0.04,0.049998,-1.7,-2.872531,-3.7,0.0000,-0.094706,-0.061970



Letzte 5 Zeilen des final_df_lagged:


,Datum,Rezession,Industrial_Production_lag_1,Manufacturing_Output_lag_1,Capital_Goods_Output_lag_1,Intermediate_Goods_Output_lag_1,Consumer_Goods_Output_lag_1,Orders_Abroad_Intermediate_Capital_lag_1,Construction_Orders_lag_1,Unemployment_Rate_lag_1,...,Domestic_Capital_Goods_Orders_lag_12,Domestic_Intermediate_Goods_Orders_lag_12,Euribor_3M_lag_12,Brent_Oil_lag_12,Production_Expectations_Manufacturing_lag_12,Business_Situation_Services_lag_12,Business_Situation_Retail_lag_12,Term_Spread_Germany_lag_12,Exports_lag_12,Imports_lag_12
396,2026-01-31,0,-0.009767,-0.011809,-0.021783,-0.007160,0.007555,0.029050,-0.034906,0.0,...,-0.054899,0.022114,-0.121,0.070688,-10.9,0.3,-4.3,-0.031115,0.008335,0.033959
397,2026-02-28,0,-0.001091,-0.006501,-0.011072,-0.004802,-0.004310,-0.085181,-0.061028,0.0,...,-0.006795,-0.013456,-0.179,-0.049522,-6.9,2.1,1.4,0.132600,0.009408,-0.000967
398,2026-03-31,0,0.000000,0.000000,-0.001013,0.005999,-0.010858,0.057223,0.058873,0.0,...,0.014665,0.014670,-0.083,-0.036584,-5.2,-5.5,-2.5,-0.082732,0.013986,-0.003524
399,2026-04-30,0,-0.007671,-0.005450,-0.011207,0.004773,-0.013187,0.048946,-0.002160,0.0,...,0.036288,-0.033316,-0.193,-0.065336,-6.4,1.3,0.2,0.190552,-0.010187,0.014806
400,2026-05-31,0,0.000000,0.000000,-0.015488,0.014185,0.018631,-0.035945,0.000000,0.1,...,-0.022939,-0.031870,-0.162,-0.055528,-1.7,-1.3,4.6,0.074259,-0.014207,-0.033695



--- Qualitätskontrolle abgeschlossen ---


**X_final und y_final erstellen**

Nach der Erstellung und Qualitätskontrolle der Lag Features wird der Datensatz für die nachfolgenden Modellierungs- und Validierungsschritte vorbereitet. Hierzu werden die zeitverzögerten Prädiktorvariablen in der finalen Feature-Matrix X_final zusammengefasst, während die binäre Zielvariable Rezession separat als Zielvektor y_final definiert wird. Die Variable Datum wird bewusst aus der Feature-Matrix ausgeschlossen, da sie nicht als Prädiktor in die Machine-Learning-Modelle eingehen soll.

Zusätzlich werden die Datumswerte separat in final_df_lagged_dates gespeichert, da sie für die anschließende zeitbasierte Kreuzvalidierung benötigt werden. Abschließend werden die Dimensionen von X_final und y_final ausgegeben, um sicherzustellen, dass Feature-Matrix und Zielvariable auf derselben Anzahl von Beobachtungen basieren.

In [34]:
# Prepare final feature matrix and target vector
X_final = final_df_lagged.drop(columns=["Datum", target_column_name])
y_final = final_df_lagged[target_column_name]

# Ensure feature names are strings
X_final.columns = X_final.columns.astype(str)

# Keep the dates for custom_cv
final_df_lagged_dates = final_df_lagged["Datum"].reset_index(drop=True)

print(f"X_final shape: {X_final.shape}")
print(f"y_final shape: {y_final.shape}")

X_final shape: (401, 336)
y_final shape: (401,)


**dentifikation der Rezessionsperioden**

Zur Vorbereitung der zeitbasierten Kreuzvalidierung werden die zusammenhängenden Rezessionsperioden anhand der binären Zielvariable Rezession identifiziert. Für jede Rezessionsphase werden Beginn, Ende und Dauer bestimmt. Die ermittelten Rezessionszeiträume dienen anschließend als Grundlage für die Konstruktion der Testfenster der benutzerdefinierten Time-Series Cross-Validation. Dadurch kann gezielt überprüft werden, ob die definierten Testperioden relevante Rezessionsereignisse enthalten


In [35]:

# Assuming 'df' and 'target_column_name' are available from previous cells
# df: original DataFrame
# target_column_name: 'Rezession'

# Check if 'df' is defined, if not, raise an informative error
if 'df' not in locals() and 'df' not in globals():
    raise NameError("DataFrame 'df' not found. Please ensure the 'Data Loading' section (Cell a51d30bb) has been executed.")

# Check if 'target_column_name' is defined, if not, raise an informative error
if 'target_column_name' not in locals() and 'target_column_name' not in globals():
    raise NameError("Variable 'target_column_name' not found. Please ensure 'Step 3 – Define Target Variable' (Cell 0e0ae819) has been executed.")

recession_data_analysis = df[['Datum', target_column_name]].copy()

recession_periods = []
current_recession_start = None

print(f"Analyzing '{target_column_name}' from {recession_data_analysis['Datum'].min().strftime('%Y-%m')} to {recession_data_analysis['Datum'].max().strftime('%Y-%m')}\n")

for i in range(len(recession_data_analysis)):
    if recession_data_analysis.loc[i, target_column_name] == 1:
        if current_recession_start is None:
            current_recession_start = recession_data_analysis.loc[i, 'Datum']
    else: # Current row is 0 (expansion)
        if current_recession_start is not None:
            # End of a recession period
            recession_end = recession_data_analysis.loc[i-1, 'Datum']
            # Calculate duration in months (approximate)
            duration_days = (recession_end - current_recession_start).days
            duration_months = round(duration_days / 30.44) + 1 # Avg days in month + 1 to include start month
            recession_periods.append({
                'start_date': current_recession_start,
                'end_date': recession_end,
                'duration_months': duration_months
            })
            current_recession_start = None

# Handle case where dataset ends with a recession
if current_recession_start is not None:
    recession_end = recession_data_analysis.loc[len(recession_data_analysis)-1, 'Datum']
    duration_days = (recession_end - current_recession_start).days
    duration_months = round(duration_days / 30.44) + 1
    recession_periods.append({
        'start_date': current_recession_start,
        'end_date': recession_end,
        'duration_months': duration_months
    })

print("Identified Recession Periods:")
if not recession_periods:
    print("No recession periods identified in the dataset.")
else:
    for r_idx, r in enumerate(recession_periods):
        print(f"  Recession {r_idx+1}: Start: {r['start_date'].strftime('%Y-%m')}, End: {r['end_date'].strftime('%Y-%m')}, Duration: {r['duration_months']} months")


# Identify expansion periods
expansion_periods = []
current_expansion_start = None

# Start of first expansion (if not starting with recession)
if recession_data_analysis.loc[0, target_column_name] == 0:
    current_expansion_start = recession_data_analysis.loc[0, 'Datum']

for i in range(len(recession_data_analysis)):
    if recession_data_analysis.loc[i, target_column_name] == 0:
        if current_expansion_start is None:
            current_expansion_start = recession_data_analysis.loc[i, 'Datum']
    else: # Current row is 1 (recession)
        if current_expansion_start is not None:
            expansion_end = recession_data_analysis.loc[i-1, 'Datum']
            duration_days = (expansion_end - current_expansion_start).days
            duration_months = round(duration_days / 30.44) + 1
            expansion_periods.append({
                'start_date': current_expansion_start,
                'end_date': expansion_end,
                'duration_months': duration_months
            })
            current_expansion_start = None

# Handle case where dataset ends with an expansion
if current_expansion_start is not None:
    expansion_end = recession_data_analysis.loc[len(recession_data_analysis)-1, 'Datum']
    duration_days = (expansion_end - current_expansion_start).days
    duration_months = round(duration_days / 30.44) + 1
    expansion_periods.append({
        'start_date': current_expansion_start,
        'end_date': expansion_end,
        'duration_months': duration_months
    })

print("\nIdentified Expansion Periods:")
if not expansion_periods:
    print("No expansion periods identified in the dataset.")
else:
    for e_idx, e in enumerate(expansion_periods):
        print(f"  Expansion {e_idx+1}: Start: {e['start_date'].strftime('%Y-%m')}, End: {e['end_date'].strftime('%Y-%m')}, Duration: {e['duration_months']} months")

# Ensure the dates in final_df_lagged are used for index mapping
# Check if 'final_df_lagged' is defined
if 'final_df_lagged' not in locals() and 'final_df_lagged' not in globals():
    raise NameError("DataFrame 'final_df_lagged' not found. Please ensure 'Step 5 – Lag Feature Engineering' (Cell ba4b50ce) has been executed.")
final_df_lagged_dates = final_df_lagged['Datum']

Analyzing 'Rezession' from 1992-01 to 2026-05

Identified Recession Periods:
  Recession 1: Start: 1992-03, End: 1993-07, Duration: 17 months
  Recession 2: Start: 2001-03, End: 2003-06, Duration: 28 months
  Recession 3: Start: 2008-02, End: 2009-04, Duration: 15 months
  Recession 4: Start: 2020-03, End: 2020-06, Duration: 4 months
  Recession 5: Start: 2022-10, End: 2023-06, Duration: 9 months
  Recession 6: Start: 2023-10, End: 2024-06, Duration: 9 months
  Recession 7: Start: 2025-04, End: 2025-09, Duration: 6 months

Identified Expansion Periods:
  Expansion 1: Start: 1992-01, End: 1992-02, Duration: 2 months
  Expansion 2: Start: 1993-08, End: 2001-02, Duration: 91 months
  Expansion 3: Start: 2003-07, End: 2008-01, Duration: 55 months
  Expansion 4: Start: 2009-05, End: 2020-02, Duration: 130 months
  Expansion 5: Start: 2020-07, End: 2022-09, Duration: 27 months
  Expansion 6: Start: 2023-07, End: 2023-09, Duration: 3 months
  Expansion 7: Start: 2024-07, End: 2025-03, Duratio

### Custom Expanding-Window Time Series Cross-Validation

Für die spätere Evaluation der Machine-Learning-Modelle wird eine benutzerdefinierte zeitbasierte Kreuzvalidierung erstellt. Im Gegensatz zu einer zufälligen Aufteilung der Beobachtungen berücksichtigt dieses Verfahren die chronologische Struktur der makroökonomischen Zeitreihe. Für jede geeignete Rezessionsperiode wird ein separates Testfenster definiert, das neben der eigentlichen Rezession zusätzlich sechs Monate vor Beginn und sechs Monate nach Ende der Rezession umfasst. Die jeweils davorliegenden Beobachtungen bilden das Trainingsfenster.

Um eine ausreichende Datengrundlage für die Modellschätzung sicherzustellen, wird eine Mindestgröße von 60 monatlichen Trainingsbeobachtungen vorausgesetzt. Rezessionsperioden, für die diese Voraussetzung nicht erfüllt ist, werden nicht als eigenständiger Cross-Validation-Fold berücksichtigt. Zusätzlich wird kontrolliert, dass Trainings- und Testdaten zeitlich nicht überlappen und dass jedes Testfenster tatsächlich mindestens eine Rezessionsbeobachtung enthält.

Die resultierenden Train-Test-Indizes werden in custom_cv gespeichert und können anschließend für die zeitgerechte Evaluation der Machine-Learning-Modelle verwendet werden.

In [36]:
import numpy as np

# The 'final_df_lagged' DataFrame is assumed to be available from Step 5.
# It contains the 'Datum' column and the lagged features, and its index is suitable for X_final and y_final.

# Check if 'final_df_lagged' is defined, if not, raise an informative error
if 'final_df_lagged' not in locals() and 'final_df_lagged' not in globals():
    raise NameError("DataFrame 'final_df_lagged' not found. Please ensure 'Step 5 – Lag Feature Engineering' (Cell ba4b50ce) has been executed.")

# Check if 'recession_periods' is defined, if not, raise an informative error
if 'recession_periods' not in locals() and 'recession_periods' not in globals():
    raise NameError("List 'recession_periods' not found. Please ensure the 'Analyzing Recession and Expansion Periods' section (Cell 6bb3ff01) has been executed.")

custom_cv = []

# Define a minimum training window size (e.g., 5 years or 60 months in the lagged data)
# This ensures models have sufficient data to learn before the first evaluation.
min_initial_train_samples = 60

# Define buffer months for validation window around recession
months_before_recession = 6  # e.g., 6 months before the recession starts
months_after_recession = 6   # e.g., 6 months after the recession ends

# Ensure the first data point in final_df_lagged is properly handled
if len(final_df_lagged) < min_initial_train_samples:
    raise ValueError("Not enough data in final_df_lagged for minimum training size.")

print(f"Constructing custom_cv with a minimum initial training size of {min_initial_train_samples} samples.")
print(f"Each validation window will include {months_before_recession} months before and {months_after_recession} months after each recession period.\n")

for r_idx, r in enumerate(recession_periods):
    rec_start_date = r['start_date']
    rec_end_date = r['end_date']

    # Map recession dates to indices in final_df_lagged_dates
    test_recession_indices_bool = (
        (final_df_lagged_dates >= rec_start_date) &
        (final_df_lagged_dates <= rec_end_date)
    )
    test_recession_indices = final_df_lagged_dates[test_recession_indices_bool].index.to_numpy()

    if len(test_recession_indices) == 0:
        # This recession period is entirely outside the range of final_df_lagged (e.g., too early in the raw data)
        # Or the recession dates do not align perfectly with the monthly data points within final_df_lagged
        print(f"  Skipping Recession {r_idx+1} (Start: {r['start_date'].strftime('%Y-%m')}) - No corresponding data in final_df_lagged or recession period falls outside processed data range.")
        continue

    # Get the exact start and end indices of the recession within final_df_lagged
    true_rec_start_idx = test_recession_indices.min()
    true_rec_end_idx = test_recession_indices.max()

    # Define the extended test window
    extended_test_start_idx = max(0, true_rec_start_idx - months_before_recession)
    extended_test_end_idx = min(len(final_df_lagged) - 1, true_rec_end_idx + months_after_recession)

    # Define the training window: all data from the beginning up to the start of the extended test window.
    train_end_idx = extended_test_start_idx - 1

    # Ensure the training window is valid and meets minimum size requirement
    if train_end_idx < min_initial_train_samples - 1: # -1 because arange goes up to (end-1), but we want end_idx to be the actual last index
        # If the training set for this recession would be too small, skip this fold.
        print(f"  Skipping Recession {r_idx+1} (Recession start: {r['start_date'].strftime('%Y-%m')}) - Training window too small ({train_end_idx+1} samples). Minimum required: {min_initial_train_samples} samples.")
        continue

    train_index = np.arange(0, train_end_idx + 1)
    test_index = np.arange(extended_test_start_idx, extended_test_end_idx + 1)

    # Basic validation of the fold
    if len(train_index) == 0 or len(test_index) == 0:
        print(f"  Skipping Recession {r_idx+1} due to empty train or test set after extending.")
        continue
    if train_index[-1] >= test_index[0]:
        print(f"  Warning: Train and test sets overlap for Recession {r_idx+1}. This should not happen with current logic. Skipping.")
        continue

    # Verify that the test window indeed contains the recession
    # 'target_column_name' is defined in cell 0e0ae819
    test_window_recession_check = final_df_lagged[target_column_name].iloc[test_index]
    if not (test_window_recession_check == 1).any():
        print(f"  Warning: Extended test window for Recession {r_idx+1} (Test Dates: {final_df_lagged_dates.iloc[test_index[0]].strftime('%Y-%m')} to {final_df_lagged_dates.iloc[test_index[-1]].strftime('%Y-%m')}) does not contain any recession observations. Skipping fold.")
        continue

    custom_cv.append((train_index, test_index))

    # Get dates for better readability in output
    train_start_date_str = final_df_lagged_dates.iloc[train_index[0]].strftime('%Y-%m')
    train_end_date_str = final_df_lagged_dates.iloc[train_index[-1]].strftime('%Y-%m')
    test_start_date_str = final_df_lagged_dates.iloc[test_index[0]].strftime('%Y-%m')
    test_end_date_str = final_df_lagged_dates.iloc[test_index[-1]].strftime('%Y-%m')

    print(f"  Fold {len(custom_cv)} (Recession {r_idx+1}):")
    print(f"    Train: {train_start_date_str} to {train_end_date_str} ({len(train_index)} samples)")
    print(f"    Test: {test_start_date_str} to {test_end_date_str} ({len(test_index)} samples)")
    print(f"    Original Recession: {r['start_date'].strftime('%Y-%m')} to {r['end_date'].strftime('%Y-%m')}")


print(f"\nGenerated {len(custom_cv)} custom cross-validation splits.")

# The variable 'custom_cv' is now ready to be used in place of 'tscv' in subsequent GridSearchCV or manual cross-validation loops.

Constructing custom_cv with a minimum initial training size of 60 samples.
Each validation window will include 6 months before and 6 months after each recession period.

  Skipping Recession 1 (Recession start: 1992-03) - Training window too small (0 samples). Minimum required: 60 samples.
  Fold 1 (Recession 2):
    Train: 1993-01 to 2000-08 (92 samples)
    Test: 2000-09 to 2003-12 (40 samples)
    Original Recession: 2001-03 to 2003-06
  Fold 2 (Recession 3):
    Train: 1993-01 to 2007-07 (175 samples)
    Test: 2007-08 to 2009-10 (27 samples)
    Original Recession: 2008-02 to 2009-04
  Fold 3 (Recession 4):
    Train: 1993-01 to 2019-08 (320 samples)
    Test: 2019-09 to 2020-12 (16 samples)
    Original Recession: 2020-03 to 2020-06
  Fold 4 (Recession 5):
    Train: 1993-01 to 2022-03 (351 samples)
    Test: 2022-04 to 2023-12 (21 samples)
    Original Recession: 2022-10 to 2023-06
  Fold 5 (Recession 6):
    Train: 1993-01 to 2023-03 (363 samples)
    Test: 2023-04 to 2024-12 (

**Kontrolle der Klassenverteilung innerhalb der Test-Folds**

Nach der Konstruktion der benutzerdefinierten zeitbasierten Kreuzvalidierung wird die Verteilung der Zielklassen innerhalb der einzelnen Test-Folds untersucht. Hierzu werden für jedes Testfenster sowohl die absolute als auch die relative Häufigkeit von Rezessionsmonaten (Rezession = 1) und Nicht-Rezessionsmonaten (Rezession = 0) bestimmt. Diese Kontrolle stellt sicher, dass die für die Modellevaluation vorgesehenen Testperioden tatsächlich Rezessionsbeobachtungen enthalten und gleichzeitig eine Bewertung gegenüber Nicht-Rezessionsperioden ermöglichen.

In [37]:
# Check if 'custom_cv' is defined, if not, raise an informative error
if 'custom_cv' not in locals() and 'custom_cv' not in globals():
    raise NameError("List 'custom_cv' not found. Please ensure the 'Custom Expanding-Window Time Series Cross-Validation' section (Cell 629ba3f9) has been executed.")

# Check if 'final_df_lagged' is defined, if not, raise an informative error
if 'final_df_lagged' not in locals() and 'final_df_lagged' not in globals():
    raise NameError("DataFrame 'final_df_lagged' not found. Please ensure 'Step 5 – Lag Feature Engineering' (Cell ba4b50ce) has been executed.")

# Check if 'target_column_name' is defined, if not, raise an informative error
if 'target_column_name' not in locals() and 'target_column_name' not in globals():
    raise NameError("Variable 'target_column_name' not found. Please ensure 'Step 3 – Define Target Variable' (Cell 0e0ae819) has been executed.")

print("Class distribution for 'Rezession' in each test fold:")
for i, (train_index, test_index) in enumerate(custom_cv):
    y_test_fold = final_df_lagged[target_column_name].iloc[test_index]
    print(f"\nFold {i+1} Test Set Class Distribution:")
    print(y_test_fold.value_counts())
    print(f"  Recession (1) percentage: {y_test_fold.value_counts(normalize=True).get(1, 0) * 100:.2f}%")
    print(f"  Non-Recession (0) percentage: {y_test_fold.value_counts(normalize=True).get(0, 0) * 100:.2f}%")

Class distribution for 'Rezession' in each test fold:

Fold 1 Test Set Class Distribution:
Rezession
1    28
0    12
Name: count, dtype: int64
  Recession (1) percentage: 70.00%
  Non-Recession (0) percentage: 30.00%

Fold 2 Test Set Class Distribution:
Rezession
1    15
0    12
Name: count, dtype: int64
  Recession (1) percentage: 55.56%
  Non-Recession (0) percentage: 44.44%

Fold 3 Test Set Class Distribution:
Rezession
0    12
1     4
Name: count, dtype: int64
  Recession (1) percentage: 25.00%
  Non-Recession (0) percentage: 75.00%

Fold 4 Test Set Class Distribution:
Rezession
1    12
0     9
Name: count, dtype: int64
  Recession (1) percentage: 57.14%
  Non-Recession (0) percentage: 42.86%

Fold 5 Test Set Class Distribution:
Rezession
1    12
0     9
Name: count, dtype: int64
  Recession (1) percentage: 57.14%
  Non-Recession (0) percentage: 42.86%

Fold 6 Test Set Class Distribution:
Rezession
0    12
1     6
Name: count, dtype: int64
  Recession (1) percentage: 33.33%
  Non-R

**Speicherung**

In [38]:
# Finalen Datensatz nach dem Feature Engineering speichern

output_file = "Final_Dataset_Feature_Engineering_1991_2026.xlsx"

final_df_lagged.to_excel(
    output_file,
    index=False
)

print(f"Datensatz erfolgreich gespeichert: {output_file}")
print("Dimension:", final_df_lagged.shape)
print(
    "Zeitraum:",
    final_df_lagged["Datum"].min(),
    "bis",
    final_df_lagged["Datum"].max()
)

Datensatz erfolgreich gespeichert: Final_Dataset_Feature_Engineering_1991_2026.xlsx
Dimension: (401, 338)
Zeitraum: 1993-01-31 00:00:00 bis 2026-05-31 00:00:00
